#### Create CBA similarity - Untreated varying, Treated fixed.

In [5]:
import pandas as pd
import numpy as np
import duckdb
import subprocess
import sys
from pathlib import Path


In [ ]:
# Paths 

main = Path("/kellogg/proj/lgg3230")
rais_aux = main / "UnionSpill/Data/RAIS_aux"

# DAta paths:


clauses_path = rais_aux / "cba_clauses_by_period.dta"
bilateral_path = rais_aux / "bilateral_connectivity_2007_2011.csv"
output_path = rais_aux / "cba_similarity_panel.dta"

In [31]:
# Per-year firm employment, sample-restricted to lagos & balanced panel                                          
firm_path = main / "UnionSpill/Data/CBA_RAIS_firm_level/cba_rais_firm_2007_2016.dta"                             
firm = pd.read_stata(                                                                                            
     str(firm_path),                                                                                              
      columns=['identificad','year','firm_emp',                                                                    
               'lagos_sample_avg','in_balanced_panel','treat_ultra'],                                              
      convert_categoricals=False                                                                                   
)                                                                                                                
firm = firm[(firm.year.between(2007, 2011)) &                                                                    
              (firm.lagos_sample_avg == 1) &
              (firm.in_balanced_panel == 1)].copy()                                                                
firm['identificad'] = firm['identificad'].astype(str).str.strip().str.zfill(14)                                  
                                                                                  

In [35]:
firm.query('lagos_sample_avg==1 & in_balanced_panel==1 & treat_ultra==1')['identificad'].nunique()

12276

In [38]:
# Wide: one row per firm, columns e07..e11                                                                      
emp = (firm.pivot_table(index='identificad', columns='year',                                                     
                          values='firm_emp', aggfunc='first')                                                      
              .rename(columns={2007:'e07', 2008:'e08', 2009:'e09',                                                 
                               2010:'e10', 2011:'e11'}))                                                           
  # Average employment for each year-pair = (emp_y + emp_{y+1}) / 2                                                
emp['ae0708'] = (emp['e07'] + emp['e08']) / 2
emp['ae0809'] = (emp['e08'] + emp['e09']) / 2                                                                    
emp['ae0910'] = (emp['e09'] + emp['e10']) / 2
emp['ae1011'] = (emp['e10'] + emp['e11']) / 2                                                                    
                  
# Treat status per firm                                                                                          
treat_status = firm.groupby('identificad')['treat_ultra'].max()
print(f"Sample firms with emp: {len(emp):,}  |  treated: {(treat_status==1).sum():,}")   

Sample firms with emp: 16,472  |  treated: 12,276


In [7]:
# loading data:
clauses = pd.read_stata(str(clauses_path), convert_categoricals=False)

bilateral = pd.read_csv(str(bilateral_path))

In [36]:
clauses.query('lagos_sample_avg==1 & in_balanced_panel==1 & treat_ultra==1')['identificad'].nunique()


12276

In [18]:
bilateral.describe()


,identificad_i,identificad_j,bilateral_conn_pw,flows_total,flows_0708,flows_0809,flows_0910,flows_1011,ratio_0708,ratio_0809,ratio_0910,ratio_1011
count,6.255900e+04,6.255900e+04,38634.000000,62559.000000,62559.000000,62559.000000,62559.000000,62559.000000,11618.000000,11651.000000,13583.000000,15259.000000
mean,1.161829e+14,1.284229e+14,0.038127,3.368228,0.780975,0.786346,0.894707,0.906201,0.040343,0.039941,0.033969,0.041610
std,2.436854e+13,3.044536e+13,0.066595,21.481195,8.974557,6.945588,5.528791,6.384663,0.083555,0.072505,0.063808,0.066726
min,1.000020e+14,1.000096e+14,0.000071,1.000000,0.000000,0.000000,0.000000,0.000000,0.000071,0.000083,0.000079,0.000077
25%,1.003603e+14,1.003603e+14,0.003454,1.000000,0.000000,0.000000,0.000000,0.000000,0.003697,0.003653,0.002907,0.003831
50%,1.029145e+14,1.130045e+14,0.017544,1.000000,0.000000,0.000000,0.000000,0.000000,0.017309,0.016713,0.013423,0.019608
75%,1.300985e+14,1.580710e+14,0.048925,2.000000,1.000000,1.000000,1.000000,1.000000,0.050000,0.050000,0.040000,0.054054
max,1.984086e+14,1.985979e+14,2.000000,3140.000000,1024.000000,961.000000,713.000000,1018.000000,2.000000,2.000000,1.666667,1.600000


In [21]:
# Generating connecitivity to treatment variab

ratios = ['ratio_0708', 'ratio_0809', 'ratio_0910', 'ratio_1011']


bilateral['bilateral_pw'] = bilateral[ratios].mean(axis=1)

bilateral[['bilateral_pw','bilateral_conn_pw']].describe()

,bilateral_pw,bilateral_conn_pw
count,38634.000000,38634.000000
mean,0.038127,0.038127
std,0.066595,0.066595
min,0.000071,0.000071
25%,0.003454,0.003454
50%,0.017544,0.017544
75%,0.048925,0.048925
max,2.000000,2.000000


In [29]:
pd.concat([bilateral['identificad_j'], bilateral['identificad_i']]).nunique()


13465

In [39]:
# Convert bilateral IDs to 14-digit strings (strip leading "1")                                                  
b = bilateral[['identificad_i','identificad_j',
                 'flows_0708','flows_0809','flows_0910','flows_1011']].copy()                                      
b['id_i'] = b['identificad_i'].astype('int64').astype(str).str.zfill(15).str[1:]
b['id_j'] = b['identificad_j'].astype('int64').astype(str).str.zfill(15).str[1:]                                 
                                                                                                                   
flow_cols = ['flows_0708','flows_0809','flows_0910','flows_1011']                                                
fwd = b.rename(columns={'id_i':'focal','id_j':'partner'})[['focal','partner']+flow_cols]                         
rev = b.rename(columns={'id_j':'focal','id_i':'partner'})[['focal','partner']+flow_cols]
pairs = pd.concat([fwd, rev], ignore_index=True)                                               

In [ ]:
pairs['some_flow'] = 

,flows_0708,flows_0809,flows_0910,flows_1011
count,125118.000000,125118.000000,125118.000000,125118.000000
mean,0.780975,0.786346,0.894707,0.906201
std,8.974521,6.945560,5.528769,6.384638
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000
max,1024.000000,961.000000,713.000000,1018.000000


In [45]:
# Merge focal's avg-emp per year-pair (inner: focal must be in sample)                                           
pairs = pairs.merge(                    
      emp[['ae0708','ae0809','ae0910','ae1011']],                                                                  
      left_on='focal', right_index=True, how='inner'                                                               
)                                                                                                                
  # Restrict partner to sample as well                                                                             
pairs = pairs[pairs['partner'].isin(emp.index)].copy()
                                                                                                                   
  # Per year-pair ratio with FOCAL firm's avg-emp as denominator
  # (NaN when no flows that year OR avg_emp == 0)                                                                  
def safe_div(num, den):                                                                                          
      return np.where((den > 0) & (num > 0), num / den, np.nan)                                                    
                                                                                                                   
pairs['r0708'] = safe_div(pairs['flows_0708'].values, pairs['ae0708'].values)                                    
pairs['r0809'] = safe_div(pairs['flows_0809'].values, pairs['ae0809'].values)
pairs['r0910'] = safe_div(pairs['flows_0910'].values, pairs['ae0910'].values)                                    
pairs['r1011'] = safe_div(pairs['flows_1011'].values, pairs['ae1011'].values)
                                                                                                                   
# Mean across non-missing year-pairs (matches MATLAB bilateral_conn_pw convention)                               
pairs['weight'] = pairs[['r0708','r0809','r0910','r1011']].mean(axis=1)                                          
pairs = pairs[pairs['weight'] > 0][['focal','partner','weight']]                                                 
                                                                                                                   
  # Filter: focal untreated, partner treated
pairs = pairs[(pairs['focal'].map(treat_status) == 0) &                                                          
                (pairs['partner'].map(treat_status) == 1)].copy()                                                  
                                          
print(f"Untreated→treated pairs (orientation-corrected): {len(pairs):,}")                                        
print(f"Untreated focal firms with at least one treated partner: "                                               
        f"{pairs['focal'].nunique():,}")      
pairs['weight'].describe()                                                   

Untreated→treated pairs (orientation-corrected): 11,471
Untreated focal firms with at least one treated partner: 2,208


count    11471.000000
mean         0.011047
std          0.031841
min          0.000072
25%          0.001050
50%          0.003160
75%          0.009145
max          1.218515
Name: weight, dtype: float64

In [47]:
pairs.head()

,focal,partner,weight
46,00037127000185,00360305266058,0.019231
47,00037226000167,00314310000180,0.002535
75,00048785004593,04928297000100,0.014925
76,00048785004593,07451885000941,0.012821
116,00059822000300,02390435000468,0.002823


In [50]:
clauses['identificad'] = clauses['identificad'].astype(str).str.strip().str.zfill(14)                            
clause_vars = [c for c in clauses.columns if c.startswith('cl_')]
                                                                                                                   
  # Treated firms' clause vectors in cba_period 2
cl2 = (clauses[(clauses['cba_period'] == 2) & (clauses['treat_ultra'] == 1)]                                     
         [['identificad'] + clause_vars]                                                                           
         .rename(columns={'identificad': 'partner'}))                                                              
cl2[clause_vars] = cl2[clause_vars].fillna(0)                                                                    
                                                                                                                   
merged = pairs.merge(cl2, on='partner', how='inner')

# ref_f = Σ_k w_fk * c_k / Σ_k w_fk         
w_total = merged.groupby('focal')['weight'].sum()                                                                
weighted = merged[clause_vars].multiply(merged['weight'].values, axis=0)
weighted['focal'] = merged['focal'].values                                                                       
ref = weighted.groupby('focal').sum().div(w_total, axis=0).reset_index()
ref = ref.rename(columns={'focal': 'identificad'})                                                               
ref.columns = ['identificad'] + [f'ref_{c}' for c in clause_vars]                                                
                                                                                                                   
print(f"Reference vectors (cba_period 2): {len(ref):,} untreated firms")                                         
ref.head()  

Reference vectors (cba_period 2): 2,208 untreated firms


,identificad,ref_cl_0,ref_cl_0not_iden,ref_cl_11des_sal,ref_cl_11iso_sal,ref_cl_11pis_sal,ref_cl_11rea_cor,ref_cl_12pag_sal,ref_cl_13rem_dsr,ref_cl_13sal_est,...,ref_cl_82dir_opo,ref_cl_82out_dis,ref_cl_82rep_sin,ref_cl_82sin_cam,ref_cl_91apl_ins,ref_cl_91des_ins,ref_cl_91mec_sol,ref_cl_91out_dis,ref_cl_91reg_par,ref_cl_91ren_res
0,00037127000185,0.0,0.0,0.000000,0.000000,1.000000,1.000000,0.000000,0.0,0.0,...,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,0.000000,4.000000,0.0,0.000000
1,00037226000167,0.0,0.0,0.000000,0.000000,1.000000,1.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.0,0.000000
2,00048785004593,0.0,0.0,0.537931,0.537931,1.000000,0.537931,1.613793,0.0,0.0,...,0.537931,0.537931,0.000000,0.537931,0.537931,0.000000,0.000000,1.848276,0.0,0.000000
3,00059822000300,0.0,0.0,0.607855,0.079091,1.041100,0.960180,1.914018,0.0,0.0,...,0.086326,0.000000,0.079091,0.621037,0.746099,0.429866,0.250383,1.180148,0.0,0.316023
4,00063960007294,0.0,0.0,0.000000,0.000000,0.603919,0.400980,0.202939,0.0,0.0,...,0.000000,0.202939,0.000000,0.000000,0.198041,0.400980,0.400980,2.608817,0.0,0.396081
